# Project Environment and Reproducible Iris Classification

**Nick Hollman**

I use this project to practice setting up a dedicated Python environment, loading local configuration and separating reusable training logic from the notebook. The example is supervised multiclass classification using scikit-learn's public Iris dataset.

My workflow uses a project folder, a local .venv, an ignored .env file, a versioned requirements.txt and a Pipeline that fits preprocessing only on training data. I can run the same training function from this notebook or from the command line.

## 1. Prepare the project

Follow the README to create .venv, install requirements and copy .env.example to .env. In VS Code, select .venv as the Python interpreter and this notebook's kernel. I verify the environment before importing project libraries.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
expected_environment = project_root / ".venv"

if not (project_root / "train.py").is_file():
    raise RuntimeError("Open the 01_iris_classification project folder and use it as the notebook working directory.")
if Path(sys.prefix).resolve() != expected_environment.resolve():
    raise RuntimeError("Select this project's .venv as the notebook kernel.")

print("Python executable", sys.executable)
print("Python version", sys.version.split()[0])
print("Project environment selected", True)


## 2. Load the shared training code

I keep training logic in train.py so the notebook and command line use the same implementation. The function reads the project's .env with python-dotenv. Existing environment variables take precedence.

RANDOM_STATE controls the repeatable split. TEST_SIZE controls the holdout proportion. ARTIFACT_DIR controls where the local model and metrics are saved. I avoid printing environment variables that could contain credentials. The .env.example file contains the safe configuration layout for another user.

In [ ]:
from train import run_training


## 3. Train and evaluate

run_training loads the public Iris dataset, splits it with stratification, fits a StandardScaler and LogisticRegression together, and evaluates on the held-out partition. The pipeline prevents the scaler from fitting on test data.

The function exports the complete pipeline and a metrics file to the ignored artifacts folder. Accuracy measures overall correctness. Macro F1 gives equal weight to each of the three flower classes.

Running the next cell trains the small example model. I interpret the resulting values only after executing it.

In [ ]:
import pandas as pd

metrics = run_training(project_root)
pd.DataFrame([metrics])


## 4. Reproduce from the terminal

From the project folder in PowerShell, I can use the environment directly.

~~~powershell
.\.venv\Scripts\python.exe train.py
~~~

This calls the same function as the notebook. The README lists the setup and dependency commands. I can recreate the environment from requirements.txt instead of copying .venv between machines.

## 5. Review before committing

I commit source code, the safe .env.example, requirements.txt and documentation. The root .gitignore excludes .env, .venv, data and artifacts.

Before staging changes, I inspect notebook source and clear outputs that could contain private records, credentials or internal identifiers. Keeping a value in .env does not remove it from a saved output.

~~~powershell
git status --short
git diff
git add -- .
git diff --cached
~~~

I inspect the staged diff and filenames before committing and pushing. For another project, I create a sibling folder with a separate environment, choose appropriate data and validation, and adapt train.py to that prediction task.